---
title: Create a list of governments
authors: 
    - Tim Sherratt
draft: true
---

The [Parliamentary Handbook API](/hansard/resources/parliamentary-handbook-api.ipynb) has lists of Prime Ministers and Ministries, but not lists of governments – ie the combination of a particular PM leading a particular party in government for an unbroken period of time. This notebook demonstrates how to generate such a list from the PMs and Ministries data.

For a similar approach that finds Ministries by date see [HOW TO: Find the Prime Minister and Ministry on a particular date](/hansard/how-to/find-ministry-by-date.ipynb).

In [23]:
import datetime

import pandas as pd
import requests

In [28]:
df_min = pd.read_json(
    "https://handbookapi.aph.gov.au/api/StatisticalInformation/Ministries",
    convert_dates=["DateStart", "DateEnd"],
)

df_min["parties"] = df_min.apply(
    lambda x: " | ".join(sorted([p["Name"] for p in x["Parties"]])), axis=1
)
df_min["party"] = df_min.apply(
    lambda x: "Labor" if "Australian Labor Party" in x["parties"] else "Non-Labor",
    axis=1,
)

In [29]:
pm_data = requests.get(
    "https://handbookapi.aph.gov.au/api/HistoricalInformation/PrimeMinisters"
).json()
pms = {pm["Phid"]: pm for pm in pm_data}
governments = []
current_pm = ""
current_party = ""
current_end = ""

# Loop through the ministries, saving a 'government' when either the PM or party changes
for mini in df_min.sort_values("Id", ascending=True).itertuples():
    # Check is either PM or party has changed
    if mini.LeaderPHID != current_pm or mini.party != current_party:
        # Set the end date of the previous government
        if current_end:
            governments[-1]["end_date"] = current_end
        # Save the new government details
        governments.append(
            {
                "phid": mini.LeaderPHID,
                "pm": pms[mini.LeaderPHID]["DisplayName"],
                "party": mini.party,
                "start_date": mini.DateStart.date().isoformat(),
                
            }
        )
    # Update values
    current_end = mini.DateEnd.date().isoformat()
    current_party = mini.party
    current_pm = mini.LeaderPHID

In [30]:
df_govs = pd.DataFrame(governments)
df_govs.head()

,phid,pm,party,start_date,end_date
0,009LR,"BARTON, the Rt. Hon. Sir Edmund, GCMG, KC",Non-Labor,1901-01-01,1903-09-24
1,009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1903-09-24,1904-04-27
2,F4R,"WATSON, the Hon. John (Chris) Christian",Labor,1904-04-27,1904-08-18
3,F4P,"REID, the Rt. Hon. Sir George Houstoun, GCB, G...",Non-Labor,1904-08-18,1905-07-05
4,009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1905-07-05,1908-11-13


In [31]:
df_govs.to_csv("governments.csv", index=False)

## List governments showing number of days in office

In [36]:
#| output: false

# Make sure dates are datetime objects
df_govs["start_date"] = pd.to_datetime(df_govs["start_date"])
df_govs["end_date"] = pd.to_datetime(df_govs["end_date"])

# The current gov will be missing an end date, fill with today's date
df_govs.fillna({"end_date": datetime.datetime.now()}, inplace=True)

# Calculate time delta
df_govs["days_in_office"] = df_govs["end_date"] - df_govs["start_date"]

# Convert time delta to days
df_govs["days_in_office"] = df_govs["days_in_office"].dt.days

# Style and display the dataframe, including a built-in bar chart
df_govs.style.bar(subset=["days_in_office"], color="#d65f5f").format(
    {"start_date": "{:%Y-%m-%d}", "end_date": "{:%Y-%m-%d}", "days_in_office": "{:,}"}
).hide()

phid,pm,party,start_date,end_date,days_in_office
009LR,"BARTON, the Rt. Hon. Sir Edmund, GCMG, KC",Non-Labor,1901-01-01,1903-09-24,996
009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1903-09-24,1904-04-27,216
F4R,"WATSON, the Hon. John (Chris) Christian",Labor,1904-04-27,1904-08-18,113
F4P,"REID, the Rt. Hon. Sir George Houstoun, GCB, GCMG, KC",Non-Labor,1904-08-18,1905-07-05,321
009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1905-07-05,1908-11-13,"1,227"
F4N,"FISHER, the Rt. Hon. Andrew",Labor,1908-11-13,1909-06-02,201
009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1909-06-02,1910-04-29,331
F4N,"FISHER, the Rt. Hon. Andrew",Labor,1910-04-29,1913-06-24,"1,152"
F4S,"COOK, the Rt. Hon. Sir Joseph, GCMG",Non-Labor,1913-06-24,1914-09-17,450
F4N,"FISHER, the Rt. Hon. Andrew",Labor,1914-09-17,1915-10-27,405


In [37]:
#| echo: false
#| column: page-inset
#| label: fig-governments-days
#| fig-cap: List of governments showing days in office

df_govs.style.bar(subset=["days_in_office"], color="#d65f5f").format(
    {"start_date": "{:%Y-%m-%d}", "end_date": "{:%Y-%m-%d}", "days_in_office": "{:,}"}
).hide()

phid,pm,party,start_date,end_date,days_in_office
009LR,"BARTON, the Rt. Hon. Sir Edmund, GCMG, KC",Non-Labor,1901-01-01,1903-09-24,996
009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1903-09-24,1904-04-27,216
F4R,"WATSON, the Hon. John (Chris) Christian",Labor,1904-04-27,1904-08-18,113
F4P,"REID, the Rt. Hon. Sir George Houstoun, GCB, GCMG, KC",Non-Labor,1904-08-18,1905-07-05,321
009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1905-07-05,1908-11-13,"1,227"
F4N,"FISHER, the Rt. Hon. Andrew",Labor,1908-11-13,1909-06-02,201
009MD,"DEAKIN, the Hon. Alfred",Non-Labor,1909-06-02,1910-04-29,331
F4N,"FISHER, the Rt. Hon. Andrew",Labor,1910-04-29,1913-06-24,"1,152"
F4S,"COOK, the Rt. Hon. Sir Joseph, GCMG",Non-Labor,1913-06-24,1914-09-17,450
F4N,"FISHER, the Rt. Hon. Andrew",Labor,1914-09-17,1915-10-27,405
